In [0]:
-- STEP 1: Active CineClub members, one row per cde_id
CREATE OR REPLACE TEMP VIEW vw_cc_active_members AS
WITH latest_snapshot AS (
    SELECT MAX(snapshot_date_id) AS snapshot_date_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
),
active_rows AS (
    SELECT
        s.cde_id,
        s.cc_user_profile_id,
        s.plan_id,
        s.renewal_status,
        s.snapshot_date_id,
        ROW_NUMBER() OVER (PARTITION BY s.cde_id ORDER BY s.activated_at_datetime DESC) AS rn
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot s
    JOIN latest_snapshot l ON s.snapshot_date_id = l.snapshot_date_id
    WHERE s.renewal_status LIKE 'Active%'
      AND s.cde_id IS NOT NULL AND s.cde_id > 0
      -- BR-025: test-account ceiling = months since Aug 2021 launch
      AND (s.recognitions_left IS NULL
           OR s.recognitions_left <= CEIL(months_between(current_date(), DATE'2021-08-01')))
)
SELECT
    cde_id,
    cc_user_profile_id,
    plan_id,
    CASE plan_id WHEN 'monthly1' THEN 'Monthly'
                 WHEN 'annual1'  THEN 'Annual'
                 WHEN 'gift-plan' THEN 'Gift'
                 ELSE plan_id END AS membership_type,
    renewal_status,
    snapshot_date_id
FROM active_rows
WHERE rn = 1;

-- STEP 2 (revised): add IS_DBOX_Flag
CREATE OR REPLACE TEMP VIEW vw_member_trans AS
SELECT
    t.CDE_ID, t.VisitDateId, t.LocationId, t.sessionskey, t.film_id,
    t.ItemSkey, t.TicketFLAG, t.ConcessionFLAG, t.Quantity, t.IS_DBOX_Flag
FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
JOIN vw_cc_active_members m ON t.CDE_ID = m.cde_id
JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l ON t.LocationId = l.LocationID
WHERE t.VisitDateId BETWEEN 20260101 AND CAST(date_format(current_date(), 'yyyyMMdd') AS INT)
  AND l.LC_Location_Type_Description = 'Cineplex Theatre'   -- BR-010
  AND t.RecRoomFLAG = 'No' AND t.STOREFLAG = 'No';          -- BR-011

-- STEP 3 (revised): carry net D-BOX seat quantity per viewing
CREATE OR REPLACE TEMP VIEW vw_member_viewings AS
WITH ticket_events AS (
    SELECT CDE_ID, VisitDateId, LocationId, sessionskey, film_id,
           SUM(Quantity) AS net_ticket_qty,
           SUM(CASE WHEN IS_DBOX_Flag = 'Yes' THEN Quantity ELSE 0 END) AS net_dbox_qty
    FROM vw_member_trans
    WHERE TicketFLAG = 'Yes'
    GROUP BY CDE_ID, VisitDateId, LocationId, sessionskey, film_id
    HAVING SUM(Quantity) > 0
)
SELECT
    e.CDE_ID, e.VisitDateId, e.LocationId, e.sessionskey, e.film_id,
    e.net_ticket_qty, e.net_dbox_qty,
    f.title_id, f.title_name,
    s.Auditorium_Experience_ID, s.Performance_Experience_ID,
    s.Is_DBOXflag,
    s.Is_70MMflag
FROM ticket_events e
LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_film f    ON e.film_id = f.film_id
LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_session s ON e.sessionskey = s.sessionskey;

-- STEP 4: Viewings per member x title
CREATE OR REPLACE TEMP VIEW vw_title_counts AS
SELECT CDE_ID, title_id, MAX(title_name) AS title_name, COUNT(*) AS viewings
FROM vw_member_viewings
WHERE title_id IS NOT NULL
GROUP BY CDE_ID, title_id;

-- STEP 5 (revised): D-BOX = viewing with net D-BOX seats > 0
CREATE OR REPLACE TEMP VIEW vw_format_counts AS
SELECT CDE_ID, format, COUNT(*) AS viewings, COUNT(DISTINCT title_id) AS distinct_titles
FROM (
    SELECT CDE_ID, title_id,
           stack(8,
               'VIP',      Auditorium_Experience_ID = 4,
               'UltraAVX', Auditorium_Experience_ID = 2,
               'IMAX',     Auditorium_Experience_ID = 3,
               '3D',       Performance_Experience_ID = 2,
               'D-BOX',    net_dbox_qty > 0,
               'ScreenX',  Auditorium_Experience_ID = 8,
               '70MM',     Is_70MMflag = 'Yes',
               '4DX',      Auditorium_Experience_ID = 7
           ) AS (format, in_format)
    FROM vw_member_viewings
)
WHERE in_format
GROUP BY CDE_ID, format;

CREATE OR REPLACE TEMP VIEW vw_member_concessions AS
SELECT
    t.CDE_ID, i.Item_Description, i.Item_Class_Description1, t.Quantity,
    CASE
        WHEN i.Item_Class_Description1 IN ('VHO-CANDY', 'CANDY', 'SMALL CANDY')
             AND UPPER(i.Item_Description) NOT LIKE 'DELIVERY%'
             AND UPPER(i.Item_Description) NOT LIKE 'VHO DELIVERY%'
             AND UPPER(i.Item_Description) NOT LIKE 'SNACK%'
             AND UPPER(i.Item_Description) NOT LIKE '%UPSIZE%'            THEN 'Candy'
        WHEN i.Item_Class_Description1 IN ('VHO-POPCORN', 'VHO-FBAPP POPCORN')
             AND UPPER(i.Item_Description) LIKE '%POPCORN%'
             AND UPPER(i.Item_Description) NOT LIKE '%DELIVERY%'
             AND UPPER(i.Item_Description) NOT LIKE '%SEASONING%'
             AND UPPER(i.Item_Description) NOT LIKE '%UPSIZE%'            THEN 'Popcorn'
        WHEN i.Item_Class_Description1 = 'VHO-DRINK'
             AND (UPPER(i.Item_Description) LIKE '%FOUNTAIN%'
                  OR UPPER(i.Item_Description) LIKE 'VHO DRINK - VIP DRINK%'
                  OR UPPER(i.Item_Description) LIKE '%REFIL%'
                  OR UPPER(i.Item_Description) LIKE '%MOVIE SIZED CUP%')
             AND UPPER(i.Item_Description) NOT LIKE '%BTL%'
             AND UPPER(i.Item_Description) NOT LIKE '%UPSIZE%'
             AND UPPER(i.Item_Description) NOT LIKE '%SWAP%'
             AND UPPER(i.Item_Description) NOT LIKE '%FEATURE MOD%'       THEN 'Fountain Drinks'
    END AS category
FROM vw_member_trans t
JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i ON t.ItemSkey = i.itemskey
WHERE t.ConcessionFLAG = 'Yes';

-- STEP 6b: Net quantity per member x category
CREATE OR REPLACE TEMP VIEW vw_concession_counts AS
SELECT CDE_ID, category, SUM(Quantity) AS net_qty
FROM vw_member_concessions
WHERE category IS NOT NULL
GROUP BY CDE_ID, category
HAVING SUM(Quantity) > 0;

-- STEP 7: Rank every category
CREATE OR REPLACE TEMP VIEW vw_ranked AS
SELECT 'Most watched - ' || title_name AS category, title_name AS detail, CDE_ID,
       viewings AS metric_value,
       RANK() OVER (PARTITION BY title_id ORDER BY viewings DESC) AS rnk
FROM vw_title_counts
WHERE title_id IN (39635, 38369, 38323, 37441, 38043, 39487, 39675)
UNION ALL
SELECT 'Most watched - any single movie', title_name, CDE_ID, viewings,
       RANK() OVER (ORDER BY viewings DESC)
FROM vw_title_counts
UNION ALL
SELECT 'Most purchased - ' || category, NULL, CDE_ID, net_qty,
       RANK() OVER (PARTITION BY category ORDER BY net_qty DESC)
FROM vw_concession_counts
UNION ALL
SELECT 'Most movies in - ' || format, NULL, CDE_ID, viewings,
       RANK() OVER (PARTITION BY format ORDER BY viewings DESC)
FROM vw_format_counts;

-- STEP 8: Preferred location per member = most visits (KPI-08 visit key), Jan 1 2026 to today, theatre scope
CREATE OR REPLACE TEMP VIEW vw_preferred_location AS
WITH member_visits AS (
    SELECT CDE_ID, VisitDateId, LocationId
    FROM vw_member_trans
    GROUP BY CDE_ID, VisitDateId, LocationId
    HAVING SUM(Quantity) > 0                      -- BR-014 refund netting
),
location_visits AS (
    SELECT CDE_ID, LocationId,
           COUNT(*)         AS location_visits,
           MAX(VisitDateId) AS last_visit_date
    FROM member_visits
    GROUP BY CDE_ID, LocationId
),
ranked_locations AS (
    SELECT CDE_ID, LocationId, location_visits,
           ROW_NUMBER() OVER (PARTITION BY CDE_ID
                              ORDER BY location_visits DESC, last_visit_date DESC, LocationId) AS rn
    FROM location_visits
)
SELECT r.CDE_ID,
       r.LocationId      AS preferred_location_id,
       l.Location_Name   AS preferred_location_name,
       r.location_visits AS preferred_location_visits
FROM ranked_locations r
LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l ON r.LocationId = l.LocationID
WHERE r.rn = 1;

-- FINAL: top 5 per category with membership type and preferred location (ties included)
SELECT r.category, r.detail, r.CDE_ID, m.membership_type, m.plan_id,
       p.preferred_location_id, p.preferred_location_name, p.preferred_location_visits,
       r.metric_value, r.rnk
FROM vw_ranked r
JOIN vw_cc_active_members m ON r.CDE_ID = m.cde_id
LEFT JOIN vw_preferred_location p ON r.CDE_ID = p.CDE_ID
WHERE r.rnk <= 5
ORDER BY r.category, r.rnk, r.CDE_ID;

-- 1. Population: rows = distinct members; split by type
SELECT membership_type, COUNT(*) AS row_count, COUNT(DISTINCT cde_id) AS members,
       MAX(snapshot_date_id) AS snapshot_used
FROM vw_cc_active_members GROUP BY membership_type;

-- 2. Date and scope: window and member coverage
SELECT COUNT(*) AS line_count, COUNT(DISTINCT CDE_ID) AS members_with_activity,
       MIN(VisitDateId) AS min_date, MAX(VisitDateId) AS max_date
FROM vw_member_trans;

-- 3. Join check: rows must equal distinct viewings; low NULL title/session
SELECT COUNT(*) AS row_count,
       COUNT(DISTINCT CDE_ID, VisitDateId, LocationId, sessionskey, film_id) AS distinct_viewings,
       SUM(CASE WHEN title_id IS NULL THEN 1 ELSE 0 END) AS null_title,
       SUM(CASE WHEN Auditorium_Experience_ID IS NULL THEN 1 ELSE 0 END) AS null_session
FROM vw_member_viewings;

-- 4. The 5 titles: names match, see how many film_ids roll up
SELECT title_id, title_name, COUNT(DISTINCT film_id) AS film_ids
FROM cpx_dataanalytics_gold.customer360.cpx_d_film
WHERE title_id IN (39635, 38369, 38323, 37441, 38043, 39487, 39675)
GROUP BY title_id, title_name;

-- 5. Concession patterns: eyeball what each category caught
SELECT category, Item_Description, Item_Class_Description1, SUM(Quantity) AS net_qty
FROM vw_member_concessions WHERE category IS NOT NULL
GROUP BY category, Item_Description, Item_Class_Description1
ORDER BY category, net_qty DESC;

-- 6. Format distribution: all 8 formats present with sensible volumes
SELECT format, COUNT(DISTINCT CDE_ID) AS members, SUM(viewings) AS viewings
FROM vw_format_counts GROUP BY format ORDER BY viewings DESC;

-- 7. Top 10 per category with preferred location: check ties and outliers
SELECT r.category, r.detail, r.CDE_ID, m.membership_type,
       p.preferred_location_id, p.preferred_location_name, p.preferred_location_visits,
       r.metric_value, r.rnk
FROM vw_ranked r
JOIN vw_cc_active_members m ON r.CDE_ID = m.cde_id
LEFT JOIN vw_preferred_location p ON r.CDE_ID = p.CDE_ID
WHERE r.rnk <= 10
ORDER BY r.category, r.rnk;

-- 8. Spot-check a winner against raw viewings (swap in the CDE_ID)
SELECT VisitDateId, LocationId, sessionskey, title_name, net_ticket_qty,
       Auditorium_Experience_ID, Performance_Experience_ID, Is_DBOXflag
FROM vw_member_viewings WHERE CDE_ID = 0 /* winner */ ORDER BY VisitDateId;


